### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="biogeographical_ancestry_prediction",
    dataset_year="2025", # source data is older, but this version was created in 2025
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="GitHub",
    original_dataset_source_download_link="https://github.com/CarolaHeinzel/BGA_Classification/blob/main/data/input_data/full_data.csv",
    download_description="""
We download the data from the GitHub repository and save it to a predefined folder.

mkdir -p local-data-warehouse/biogeographical_ancestry_prediction/ && curl -L -o local-data-warehouse/biogeographical_ancestry_prediction/filter_population.xlsx https://raw.githubusercontent.com/CarolaHeinzel/BGA-Classification/main/datat/filtered_population_eur_update.xlsx
""",
    # References
    academic_reference_bibtex=r"""@article{heinzel2025advancing,
  title={Advancing biogeographical ancestry predictions through machine learning},
  author={Heinzel, Carola Sophia and Purucker, Lennart and Hutter, Frank and Pfaffelhuber, Peter},
  journal={Forensic Science International: Genetics},
  volume={79},
  pages={103290},
  year={2025},
  publisher={Elsevier}
}
@article{ruiz2023development,
  title={Development and evaluations of the ancestry informative markers of the VISAGE Enhanced Tool for Appearance and Ancestry},
  author={Ruiz-Ram{\'\i}rez, Jorge and de La Puente, M and Xavier, Catarina and Ambroa-Conde, Adri{\'a}n and {\'A}lvarez-Dios, J and Freire-Aradas, A and Mosquera-Miguel, Ana and Ralf, Arwin and Amory, Christina and Katsara, Maria Alexandra and others},
  journal={Forensic Science International: Genetics},
  volume={64},
  pages={102853},
  year={2023},
  publisher={Elsevier}
}
@article{xavier2020development,
  title={Development and validation of the VISAGE AmpliSeq basic tool to predict appearance and ancestry from DNA},
  author={Xavier, Catarina and de la Puente, Maria and Mosquera-Miguel, Ana and Freire-Aradas, Ana and Kalamara, Vivian and Vidaki, Athina and Gross, Theresa E and Revoir, Andrew and Po{\'s}piech, Ewelina and Kartasi{\'n}ska, Ewa and others},
  journal={Forensic Science International: Genetics},
  volume={48},
  pages={102336},
  year={2020},
  publisher={Elsevier}
}
""",
    academic_reference_bibtex_key="heinzel2025advancing,ruiz2023development,xavier2020development",
    license="None", # Data from GitHub does not have a license, source is Apache-2.0 license (I think)
    data_tags=["IID"],
    curation_comments="""
We use this dataset as one of the most recent example of a machine learning task based on the Human Genome project.
We take the targets from the paper by Heinzel et al. (2025) and only rename the targets to be standardized and more concise.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Population",
    problem_type="multiclass_classification",
    objective_metric_name="log_loss",
    stratify_on="Population",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_excel(f"{dataset_mold.path}/filter_population.xlsx")
df = df.drop(
    columns=[
        "Unnamed: 1",
        "SNP-Indel with complex genotype listing in VCF (not compiled)",
        "Unnamed: 3",
        "Unnamed: 4",
        "ID",
    ],
)

column_map = {
    "Iberian population in Spain": "Spain (Iberian)",
    "Toscani in Italia": "Italy (Toscani)",
    "Finnish in Finland": "Finland (Finnish)",
    "Utah Residents (CEPH) with N & W European ancestry": "Utah (CEPH, N/W European ancestry)",
    "British in England and Scotland": "UK (England & Scotland, British)",
    "13. Italy - Sardinian": "Italy (Sardinian)",
    "11. France - French": "France (French)",
    "Turkey": "Turkey",
    "09. Russia - Russian": "Russia (Russian)",
    "10. France - French Basque": "France (Basque)"
}
df["Population"] = df["Population"].map(column_map)

for col in df.columns:
    df[col] = df[col].astype("category")
    
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 635
Columns: 105
Use sampling: False (sample size: 635)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['rs1398461', 'rs17287498', 'rs12629397', 'rs6504633', 'rs392461', 'rs393953', 'rs408046', 'rs2737126', 'rs5030240', 'rs2585339']
Rows remaining as candidates after top-10 filter: 0 (of 635)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,Population,rs1024124,rs1040934,rs1074689,rs10764919,rs10954737,rs10962599,rs1150911,rs11960137,rs1197062,rs12142199,rs1229984,rs12405776,rs12498138,rs12594144,rs12629397,rs12880237,rs12913832,rs1317026,rs13280988,rs1371048,rs1382568,rs1398461,rs1426654,rs1495085,rs1545397,rs1567803,rs1592672,rs166054,rs16891982,rs17086288,rs17130385,rs17287498,rs1757928,rs17625895,rs1796048,rs182857716,rs1871534,rs1924381,rs2024566,rs2026999,rs2156208,rs2196051,rs234623,rs2375771,rs2387842,rs2472304,rs2585339,rs2605361,rs262037,rs2715883,rs2737126,rs2789823,rs2814778,rs2835133,rs310362,rs367953206,rs3737576,rs3751050,rs3827760,rs3844336,rs3852253,rs3857620,rs3862700,rs392461,rs393953,rs408046,rs4308478,rs4465645,rs4471745,rs4540055,rs4657449,rs4737753,rs487750,rs5030240,rs556365,rs6088466,rs6437783,rs6496996,rs6504633,rs6588145,rs6701640,rs6754311,rs6894681,rs6933094,rs7151991,rs7171818,rs7252391,rs7594173,rs776912,rs7816786,rs7975017,rs7989291,rs8072587,rs809540,rs848461,rs881929,rs914468,rs9467370,rs9479657,rs9522149,rs9817359,rs9845503,rs9847307,rs9899480
0,Russia (Russian),CT,CT,AA,AA,TT,CC,GT,CC,GT,AA,CC,CC,GG,CC,TT,GG,GG,CC,AG,TT,CG,AG,AA,AA,AA,CT,GT,GG,GG,TT,GG,AC,AA,AA,CC,AA,GG,CC,AA,TT,CC,AG,AG,CG,CG,AG,GT,GT,CC,AA,CT,AA,TT,CC,TT,TT,TT,TT,AA,AA,AA,GG,TT,AG,AG,AG,TT,TT,GG,AC,GG,CC,TT,CC,AT,GG,TT,GG,CT,GG,AC,CC,AA,TT,GG,AA,AG,AG,AC,CT,CC,AG,GG,TT,CT,TT,CG,CC,CC,CC,GG,AG,TT,CC
1,"UK (England & Scotland, British)",CT,TT,AT,AA,CT,CT,GG,CC,TT,AA,CC,CC,GG,CC,GT,AA,AG,CC,AA,GG,CC,GG,AA,AA,AA,TT,TT,GG,GG,TT,GG,CT,AA,CC,CC,AA,GG,CC,AG,CT,CC,AA,AG,CC,CC,GG,GT,GT,CT,GG,CT,AA,TT,CC,CT,TT,TT,TT,AA,AA,AA,GG,TT,CG,AA,AT,TT,TT,GG,AA,GG,CC,CT,CC,GG,GG,TT,AG,TT,GT,AC,CT,AA,TT,AG,AG,AA,AG,AC,TT,CT,AC,GG,TT,CT,GT,CG,CC,CC,CC,GG,AA,TT,CC
2,Italy (Toscani),CT,CT,AA,AA,TT,TT,GT,CC,TT,AA,CT,CC,GG,AC,CG,GG,AA,CC,AA,GG,CG,GG,AA,AA,AA,CT,TT,GG,GG,TT,GG,CT,AC,CC,CC,AA,GG,CT,GG,TT,CT,AA,AG,CG,CT,AA,TT,GG,CC,AA,CG,AA,TT,CC,TT,TT,TT,TT,AA,AG,AG,GG,TT,CG,AG,GG,CT,TT,GG,AT,GG,CT,CC,CG,GT,AG,TT,AG,TT,TT,CC,CC,AA,CT,AG,TT,AA,AA,AT,CT,CC,CG,CG,GT,CT,TT,CG,CC,CT,CC,GG,AA,TT,CC
3,"UK (England & Scotland, British)",CT,CT,AA,AA,TT,TT,GT,CC,TT,AA,CC,CC,GG,CC,TT,AG,AG,CC,AA,GG,CC,GG,AA,AA,AA,TT,TT,GG,GG,TT,GG,AC,AA,AA,CC,AA,GG,CC,GG,TT,CT,AA,GG,CC,CT,AG,TT,TT,CC,AA,CC,AA,TT,AC,CT,TT,CT,CT,AA,AA,AA,GG,TT,CC,AG,AA,TT,TT,GG,AA,GG,CC,CT,CC,AT,GG,TT,AG,CT,GT,CC,CT,AA,TT,GG,GG,AG,AA,AA,TT,CC,AG,GG,CT,CC,GG,GG,CC,CC,CC,GG,AA,TT,CT
4,"UK (England & Scotland, British)",CT,TT,TT,GG,TT,TT,CG,CG,TT,AA,CC,CC,GG,CC,GT,GG,GG,CC,AG,GG,GG,AG,AA,AA,AT,TT,TT,GG,GG,TT,GG,TT,AC,AC,CT,AA,GG,CC,GG,CT,CC,AG,GG,GG,TT,AA,TT,GT,CC,AA,GG,AA,TT,CC,CT,TT,TT,TT,AA,AA,AA,GG,TT,GG,GG,GG,TT,TT,GG,AT,GG,CC,CC,CG,AT,AG,CT,AG,TT,TT,CC,CT,AA,TT,AG,GT,GG,AA,AA,CT,CC,AC,GG,TT,CT,GT,GG,CC,CC,CC,GG,AT,TT,CC


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Population,category,0.0,0.0,10.0,"Spain (Iberian), Italy (Toscani), Utah (CEPH, N/W European ancestry), Finland (Finnish), UK (England & Scotland, British), France (French), Turkey, Italy (Sardinian), Russia (Russian), France (Basque)"
1,rs1024124,category,0.0,0.0,3.0,"CT, CC, TT"
2,rs1040934,category,0.0,0.0,3.0,"TT, CT, CC"
3,rs1074689,category,0.0,0.0,6.0,"AA, AT, AC, TT, CT, CC"
4,rs10764919,category,0.0,0.0,3.0,"AG, AA, GG"
5,rs10954737,category,0.0,0.0,3.0,"TT, CT, NN"
6,rs10962599,category,0.0,0.0,3.0,"TT, CT, CC"
7,rs1150911,category,0.0,0.0,6.0,"GT, GG, CG, CT, TT, CC"
8,rs11960137,category,0.0,0.0,3.0,"CC, CG, GG"
9,rs1197062,category,0.0,0.0,3.0,"TT, GT, GG"


In [6]:
# Numeric Feature Statistics
numeric_stats

'No numeric features to summarize.'

In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column      rank                                                  
Population  1                        Spain (Iberian)    107  16.85
            2                        Italy (Toscani)    107  16.85
            3     Utah (CEPH, N/W European ancestry)     99  15.59
            4                      Finland (Finnish)     99  15.59
            5       UK (England & Scotland, British)     91  14.33
rs1024124   1                                     CT    321  50.55
            2                                     CC    193  30.39
            3                                     TT    121  19.06
rs1040934   1                                     TT    378  59.53
            2                                     CT    212  33.39
            3                                     CC     45   7.09
rs1074689   1                                     AA    262  41.26
            2                                     AT    232  36.54
            3                                     AC     59   9.29
            4                                     TT     57   8.98
            5                                     CT     23   3.62
rs10764919  1                                     AG    284  44.72
            2                                     AA    234  36.85
            3                                     GG    117  18.43
rs10954737  1                                     TT    578  91.02
            2                                     CT     47   7.40
            3                                     NN     10   1.57
rs10962599  1                                     TT    323  50.87
            2                                     CT    255  40.16
            3                                     CC     57   8.98
rs1150911   1                                     GT    195  30.71
            2                                     GG    149  23.46
            3                                     CG     99  15.59
            4                                     CT     92  14.49
            5                                     TT     74  11.65
rs11960137  1                                     CC    469  73.86
            2                                     CG    149  23.46
            3                                     GG     17   2.68
rs1197062   1                                     TT    579  91.18
            2                                     GT     55   8.66
            3                                     GG      1   0.16
rs12142199  1                                     AA    406  63.94
            2                                     AG    197  31.02
            3                                     GG     31   4.88
            4                                     NN      1   0.16
rs1229984   1                                     CC    584  91.97
            2                                     CT     49   7.72
            3                                     TT      2   0.31
rs12405776  1                                     CC    605  95.28
            2                                     CT     30   4.72
rs12498138  1                                     GG    550  86.61
            2                                     AG     85  13.39
rs12594144  1                                     CC    474  74.65
            2                                     AC    152  23.94
            3                                     AA      9   1.42
rs12629397  1                                     GT    274  43.15
            2                                     TT    187  29.45
            3                                     GG    111  17.48
            4                                     CG     34   5.35
            5                                     CT     26   4.09
rs12880237  1                                     AG    288  45.35
            2                                     GG    271  42.68
            3                                     AA     76  11.97
rs12913832  1              

In [8]:
# Target Distribution
target_df

,count,pct
Population,,
Spain (Iberian),107,16.85
Italy (Toscani),107,16.85
"Utah (CEPH, N/W European ancestry)",99,15.59
Finland (Finnish),99,15.59
"UK (England & Scotland, British)",91,14.33
France (French),28,4.41
Turkey,28,4.41
Italy (Sardinian),28,4.41
Russia (Russian),25,3.94


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=20, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

Using Stratified IID splits.


## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to biogeographical_ancestry_prediction/019d7374-b899-714c-9372-37a48f580b0f
019d7374-b899-714c-9372-37a48f580b0f
bab4155061902f0f65763080f1103c2149ec7325f690d35ac845c732dc9ad6dd
